# Second batch

In [1]:

import sqlite3
import random

# Database paths
source_db = r"C:\Users\alrazz\Documents\Hybrid SP_ID annotation\Combined db\Combined_single.db"

previous_sample_db = r"C:\Users\alrazz\Desktop\annotator_pilot_250_for_annotation_V2.db"

output_db = r"C:\Users\alrazz\Desktop\annotator_pilot_250_for_annotation_V3.db"

sample_size = 250

# Connect to databases
source_conn = sqlite3.connect(source_db)
previous_conn = sqlite3.connect(previous_sample_db)

source_cursor = source_conn.cursor()
previous_cursor = previous_conn.cursor()

# Identify tables
source_cursor.execute("""
    SELECT name FROM sqlite_master
    WHERE type='table' AND name NOT LIKE 'sqlite_%'
""")
source_tables = [row[0] for row in source_cursor.fetchall()]

previous_cursor.execute("""
    SELECT name FROM sqlite_master
    WHERE type='table' AND name NOT LIKE 'sqlite_%'
""")
previous_tables = [row[0] for row in previous_cursor.fetchall()]

print("Source tables:", source_tables)
print("Previous sample tables:", previous_tables)

# Assuming one main table in each database
source_table = source_tables[0]
previous_table = previous_tables[0]

# Get previously sampled IDs
previous_cursor.execute(f'SELECT id FROM "{previous_table}"')
previous_ids = {row[0] for row in previous_cursor.fetchall()}

print(f"Previously sampled IDs: {len(previous_ids)}")

# Retrieve all source records
source_cursor.execute(f'SELECT * FROM "{source_table}"')
columns = [desc[0] for desc in source_cursor.description]
all_records = source_cursor.fetchall()

# Identify column positions
id_index = columns.index("id")
tags_index = columns.index("Tags")

# Excluded tags
excluded_tags = ["JNK", "MLPT", "NSFW"]

# Filter eligible records
remaining_records = []

excluded_by_id = 0
excluded_by_tags = 0

for row in all_records:

    # Exclude previously sampled IDs
    if row[id_index] in previous_ids:
        excluded_by_id += 1
        continue

    # Exclude records containing any unwanted tag
    tags = str(row[tags_index] or "").upper()

    if any(tag in tags for tag in excluded_tags):
        excluded_by_tags += 1
        continue

    remaining_records.append(row)

print(f"Excluded due to previous sampling: {excluded_by_id}")
print(f"Excluded due to unwanted tags: {excluded_by_tags}")
print(f"Eligible records remaining: {len(remaining_records)}")

# Ensure enough records are available
if len(remaining_records) < sample_size:
    raise ValueError(
        f"Only {len(remaining_records)} eligible records remain. "
        f"Cannot sample {sample_size}."
    )

# Random sampling
random.seed(42)
new_sample = random.sample(remaining_records, sample_size)

# Create output database
output_conn = sqlite3.connect(output_db)
output_cursor = output_conn.cursor()

# Recreate basic table structure
source_cursor.execute(f'PRAGMA table_info("{source_table}")')
schema = source_cursor.fetchall()

column_definitions = []

for col in schema:
    name = col[1]
    col_type = col[2] or ""
    not_null = "NOT NULL" if col[3] else ""
    primary_key = "PRIMARY KEY" if col[5] else ""

    column_definitions.append(
        f'"{name}" {col_type} {primary_key} {not_null}'
    )

create_query = (
    f'CREATE TABLE "{source_table}" '
    f'({", ".join(column_definitions)})'
)

output_cursor.execute(create_query)

# Insert sampled records
quoted_columns = ", ".join(f'"{col}"' for col in columns)
placeholders = ", ".join(["?"] * len(columns))

insert_query = (
    f'INSERT INTO "{source_table}" '
    f'({quoted_columns}) VALUES ({placeholders})'
)

output_cursor.executemany(insert_query, new_sample)

output_conn.commit()

# Verification
output_cursor.execute(f'SELECT COUNT(*) FROM "{source_table}"')
print("\nFinal sample size:", output_cursor.fetchone()[0])

output_cursor.execute(f'SELECT id, Tags FROM "{source_table}"')
final_records = output_cursor.fetchall()

new_ids = {row[0] for row in final_records}

tag_violations = [
    row for row in final_records
    if any(tag in str(row[1] or "").upper()
           for tag in excluded_tags)
]

print("Overlap with previous sample:", len(new_ids & previous_ids))
print("Unique IDs:", len(new_ids))
print("Records containing excluded tags:", len(tag_violations))

print("\nSaved successfully to:")
print(output_db)

# Close connections
source_conn.close()
previous_conn.close()
output_conn.close()


Source tables: ['texts']
Previous sample tables: ['annotator_pilot_250_for_annotation']
Previously sampled IDs: 250
Excluded due to previous sampling: 251
Excluded due to unwanted tags: 1922
Eligible records remaining: 4021

Final sample size: 250
Overlap with previous sample: 0
Unique IDs: 250
Records containing excluded tags: 0

Saved successfully to:
C:\Users\alrazz\Desktop\annotator_pilot_250_for_annotation_V3.db


In [2]:

import sqlite3

# Input and output database paths
input_db = r"C:\Users\alrazz\Desktop\annotator_pilot_250_for_annotation_V3.db"

output_db = r"C:\Users\alrazz\Desktop\annotator_pilot_250_for_annotation_V3_cleaned.db"

# Connect to databases
source_conn = sqlite3.connect(input_db)
output_conn = sqlite3.connect(output_db)

source_cursor = source_conn.cursor()
output_cursor = output_conn.cursor()

# Identify the table
source_cursor.execute("""
    SELECT name FROM sqlite_master
    WHERE type='table' AND name NOT LIKE 'sqlite_%'
""")

tables = [row[0] for row in source_cursor.fetchall()]
table_name = tables[0]

# Copy the database structure and data
source_cursor.execute(f'PRAGMA table_info("{table_name}")')
schema = source_cursor.fetchall()

# Get all columns
columns = [col[1] for col in schema]

# Columns to clear
columns_to_clear = [
    "Turku_NLP",
    "Turku_NLP_sub",
    "Tags"
]

# Verify columns exist
for col in columns_to_clear:
    if col not in columns:
        raise ValueError(f"Column '{col}' not found!")

# Recreate basic table structure
column_definitions = []

for col in schema:
    name = col[1]
    col_type = col[2] or ""
    not_null = "NOT NULL" if col[3] else ""
    primary_key = "PRIMARY KEY" if col[5] else ""

    column_definitions.append(
        f'"{name}" {col_type} {primary_key} {not_null}'
    )

create_query = (
    f'CREATE TABLE "{table_name}" '
    f'({", ".join(column_definitions)})'
)

output_cursor.execute(create_query)

# Retrieve all records
source_cursor.execute(f'SELECT * FROM "{table_name}"')
records = source_cursor.fetchall()

# Clear specified columns
cleaned_records = []

for row in records:
    row = list(row)

    for col in columns_to_clear:
        row[columns.index(col)] = None

    cleaned_records.append(tuple(row))

# Insert cleaned records
quoted_columns = ", ".join(f'"{col}"' for col in columns)
placeholders = ", ".join(["?"] * len(columns))

insert_query = (
    f'INSERT INTO "{table_name}" '
    f'({quoted_columns}) VALUES ({placeholders})'
)

output_cursor.executemany(insert_query, cleaned_records)

# Save changes
output_conn.commit()

# Verification
output_cursor.execute(f'SELECT COUNT(*) FROM "{table_name}"')
print("Total records:", output_cursor.fetchone()[0])

for col in columns_to_clear:
    output_cursor.execute(
        f'SELECT COUNT(*) FROM "{table_name}" '
        f'WHERE "{col}" IS NOT NULL'
    )
    print(f"Non-empty values in {col}:", output_cursor.fetchone()[0])

print("\nCleaned database saved to:")
print(output_db)

# Close connections
source_conn.close()
output_conn.close()


Total records: 250
Non-empty values in Turku_NLP: 0
Non-empty values in Turku_NLP_sub: 0
Non-empty values in Tags: 0

Cleaned database saved to:
C:\Users\alrazz\Desktop\annotator_pilot_250_for_annotation_V3_cleaned.db
